## Run SUMMA model ##

Run the SUMMA base model, with the model runs managed by Snakemake

NOTE that the file manager will need to be updated for this workflow to run!!!
Further testing is needed.


### Write snakemake file ###

This block of code writes the snakemake file

In [29]:
%%writefile ../rules/run_pysumma_snakemake.smk
"""

Snakemake file to run the base SUMMA simulations.

The model simulation is chunked by GRU to allow for parallelization on a cluster.

The chunks of GRUs are defined by the user

"""

from pathlib import Path
import sys
import pysumma as ps
# Import local packages
sys.path.append(str(Path('../').resolve()))

sys.path.append('../')
from scripts import gpep_to_summa_utils as gts_utils
#config = gts_utils.resolve_paths(config,log_config = False)

# Resolve all file paths and directories in the config file
config['file_manager'] = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/settings/fileManager.txt'
config['summa_output_dir'] = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output'
config['attributes_nc'] = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/settings/attributes.nc'
config['gru_chunk_size'] = 8
config['case_name'] = 'tuolumne'
config['run_suffix'] = 'base'

base_forcing_dir = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/forcing/rf_dynamic_high_predictors'


# UPDATE LOCAL SUMMA PATH
config['summa_exe'] = '/Users/drc858/GitHub/summa/bin/summa.exe'


# Generate GRU start and count
num_grus = gts_utils.calc_num_grus(config['attributes_nc'])
gru_chunk_strings = gts_utils.generate_gru_start_and_count(num_grus, chunk_size=config['gru_chunk_size'])

rule run_summa_base_simulations:
    input:
        expand(Path(config['summa_output_dir'],f"{config['case_name']}_{config['run_suffix']}_{{gru_chunk}}_timestep.nc"),gru_chunk=gru_chunk_strings)

rule run_summa_in_gru_chunks:
    input:
        file_manager = Path(config['file_manager'])
    output:
        summa_chunked_output = Path(config['summa_output_dir'],f"{config['case_name']}_{config['run_suffix']}_{{gru_chunk}}_timestep.nc")
    params:
        summa_exe = config['summa_exe'],
        gru_start = lambda wildcards: gts_utils.extract_gru_int({wildcards.gru_chunk}),
        gru_count = config['gru_chunk_size']
    run:
        sim = ps.Simulation(params.summa_exe,input.file_manager)
        sim.run(run_suffix=config['run_suffix'], startGRU=params.gru_start, countGRU=params.gru_count, write_config=False)


        

Overwriting ../rules/run_pysumma_snakemake.smk


In [5]:
! snakemake --unlock -s ../rules/run_pysumma_snakemake.smk #--configfile ../config/gpep_to_summa_tuolumne_config.yaml
! snakemake -s ../rules/run_pysumma_snakemake.smk --cores 8 #--configfile ../config/gpep_to_summa_tuolumne_config.yaml


^C


In [28]:
%%writefile ../rules/run_fsca_simulations.smk
"""

Snakemake file to calculate fSCA based on SUMMA model simulations.

"""

from pathlib import Path
import sys
# Import local packages
sys.path.append(str(Path('../').resolve()))
sys.path.append(str(Path('../../').resolve()))
sys.path.append(str(Path('../../snow_dist').resolve()))
from scripts import ss_utils
from snow_dist import summa_simulation

# Resolve all file paths and directories in the config file
config['summa_output_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/output'
config['working_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/output'
config['summa_interim_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/interim'
config['fsca_output_dir'] =  '/Users/drc858/Data/gpep/RF_ens/summa/output/fsca'
config_file_path = '/Users/drc858/GitHub/snow_dist/settings/config_summa_model_tuolumne.yaml'

#Read first raw forcing file for easymore remapping
summa_result_filepaths = list(Path(config['summa_output_dir']).glob('*.nc'))
summa_filenames = [Path(filepath).name for filepath in summa_result_filepaths]

fsca_simulation = summa_simulation.SummaSimulation(config_file_path)

# Prepare summa model results for fsca
# This includes summing the input and output snowpack variables
rule run_fsca_model:
    input:
        expand(Path(config['summa_interim_dir'], "{filenames}"))

rule prepare_fsca_model_input:
    input:
        summa_result = Path(config['summa_output_dir'],"{filenames}")
    output:
        fsca_input = Path(config['summa_interim_dir'],"{filenames}")
    run:
        summa_ds = xr.open_dataset(input.input_summa_result)
        fsca_ds = summa_fsca_model.prepare_summa_fsca_input(summa_ds)
        fsca_ds.to_netcdf(output.fsca_input)

# Run fsca settings
rule run_fsca_simulations:
    input:
        prepped_files = expand(Path(config['summa_interim_dir'], "{filenames}"), filenames=summa_filenames)
    output:
        fsca_result = Path(config['fsca_output_dir'], "{filenames}")
    run:
        sim = summa_simulation.SummaSimulation(config)
        summa_fsca_model.run_summa_fsca_model(input.prepped_files,output.fsca_result)



Overwriting ../rules/run_fsca_simulations.smk


In [16]:
#-s ../rules/run_pysumma_snakemake.smk --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml 
! snakemake -s ../rules/run_fsca_simulations.smk --cores 8 --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml

ImportError in file /Users/drc858/GitHub/gpep_to_summa_snakemake/workflow/rules/run_fsca_simulations.smk, line 13:
cannot import name 'ss_utils' from 'scripts' (unknown location)
  File "/Users/drc858/GitHub/gpep_to_summa_snakemake/workflow/rules/run_fsca_simulations.smk", line 13, in <module>


In [36]:
%%writefile ../rules/run_pysumma_bow.smk
"""

Snakemake file to run the base SUMMA simulations.

The model simulation is chunked by GRU to allow for parallelization on a cluster.

The chunks of GRUs are defined by the user

"""

from pathlib import Path
import sys
import pysumma as ps
# Import local packages
sys.path.append(str(Path('../').resolve()))

sys.path.append('../')
from scripts import gpep_to_summa_utils as gts_utils

# UPDATE LOCAL SUMMA PATH
config['summa_exe'] = '/Users/dcasson/GitHub/summa/bin/summa.exe'
config['summa_forcing_dir'] = Path('/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/forcing/')

# Resolve all file paths and directories in the config file
config['file_manager'] = '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/settings/fileManager.txt'
config['summa_output_dir'] = '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/output'
config['attributes_nc'] = '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/attributes.nc'
config['case_name'] = 'bow_above_banff'
config['run_suffix'] = 'base'


ens_set, _ = gts_utils.build_ensemble_list(config['summa_forcing_dir'])
file_paths = gts_utils.list_files_in_subdirectory(config['summa_forcing_dir'])
ens = list(ens_set)

#Filter files remove those that end in .txt
file_paths = [file for file in file_paths if not file.endswith('.txt')]

forcing_file_list = []
run_config = {}
for ens_member in ens:
    # write a new file containing the forcing files which correspond to the ensemble member
    forcing_list_file = f'{config["summa_forcing_dir"]}/forcing_files_{ens_member}.txt'
    with open(forcing_list_file, 'w') as f:
        for file in file_paths:
            #if ensemble member in first three characters of file name
            file_path = Path(file)
            complete_path = Path(config['summa_forcing_dir'],file_path).resolve()
            if ens_member in str(file):
                f.write(f'{complete_path}.nc\n')
            temp_config = {ens_member: {'forcing_file_list': forcing_list_file}}
    run_config.update(temp_config)
    forcing_file_list.append(forcing_list_file)

print(run_config)

sim = ps.Ensemble(params.summa_exe,config, input.file_manager)
sim.run()

rule run_summa_base_simulations:
    input:
        expand(Path(config['summa_output_dir'],f"{config['case_name']}_{{ens_member}}_timestep.nc"),ens_member=ens)
"""
rule run_summa_ensemble_simulations:
    input:
        file_manager = Path(config['file_manager']),
        forcing_file = lambda wildcards: forcing_file_list[ens.index(wildcards.ens_member)]
    output:
        summa_chunked_output = Path(config['summa_output_dir'],f"{config['case_name']}_{{ens_member}}_timestep.nc")
    params:
        summa_exe = config['summa_exe'],
        run_suffix = lambda wildcards: wildcards.ens_member,
    run:
        sim = ps.Ensemble(params.summa_exe,config, input.file_manager)
        #sim.force_file_list = str(input.forcing_file)
        #print(sim.force_file_list)
        #print(str(params.run_suffix))
        sim.run(run_suffix=str(params.run_suffix), write_config=False)
    """

Overwriting ../rules/run_pysumma_bow.smk


In [37]:
! snakemake --unlock -s ../rules/run_pysumma_bow.smk #--configfile ../config/gpep_to_summa_tuolumne_config.yaml
! snakemake -s ../rules/run_pysumma_bow.smk --cores 8 #--configfile ../config/gpep_to_summa_tuolumne_config.yaml


KeyError in file /Users/dcasson/GitHub/gpep_to_summa_snakemake/workflow/rules/run_pysumma_bow.smk, line 43:
'summa_forcing_dir'
  File "/Users/dcasson/GitHub/gpep_to_summa_snakemake/workflow/rules/run_pysumma_bow.smk", line 43, in <module>
KeyError in file /Users/dcasson/GitHub/gpep_to_summa_snakemake/workflow/rules/run_pysumma_bow.smk, line 43:
'summa_forcing_dir'
  File "/Users/dcasson/GitHub/gpep_to_summa_snakemake/workflow/rules/run_pysumma_bow.smk", line 43, in <module>


In [ ]:
from pathlib import Path
import sys
import pysumma as ps
# Import local packages
sys.path.append(str(Path('../').resolve()))

sys.path.append('../')
from scripts import gpep_to_summa_utils as gts_utils


config = {}
# UPDATE LOCAL SUMMA PATH
config['summa_exe'] = '/Users/dcasson/GitHub/summa/bin/summa.exe'
config['summa_forcing_dir'] = Path('/Users/dcasson/Data/gpep/chena/forcing/summa/')

# Resolve all file paths and directories in the config file
config['file_manager'] = '/Users/dcasson/Data/yukon_esp/summa/settings/fileManagerSWE.txt'
config['summa_output_dir'] = '/Users/dcasson/Data/yukon_esp/summa_output/'
config['attributes_nc'] = '/Users/dcasson/Data/yukon_esp/summa/settings/attributes.nc'
config['case_name'] = 'chena'
config['run_suffix'] = 'base'

ens_set, _ = gts_utils.build_ensemble_list(config['summa_forcing_dir'])
file_paths = gts_utils.list_files_in_subdirectory(config['summa_forcing_dir'])
ens = list(ens_set)

#Filter files remove those that end in .txt
file_paths = [file for file in file_paths if not file.endswith('.txt')]

forcing_file_list = []
for ens_member in ens:
    # write a new file containing the forcing files which correspond to the ensemble member
    forcing_list_file = f'{config["summa_forcing_dir"]}/forcing_files_{ens_member}.txt'
    with open(forcing_list_file, 'w') as f:
        for file in file_paths:
            #if ensemble member in first three characters of file name
            file_path = Path(file)
            complete_path = Path(config['summa_forcing_dir'],file_path).resolve()
            if ens_member in str(file):
                f.write(f'{file_path}.nc\n')
            temp_config = {ens_member: {'forcing_file_list': forcing_list_file}}
    forcing_file_list.append(forcing_list_file)

run_config = {
    'manager
}

managers_to_run = {
    'file_manager': ['/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/settings/fileManager001.txt',
                      '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/settings/fileManager002.txt',
                      '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/settings/fileManager003.txt',
                      '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/settings/fileManager004.txt',
                      '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/settings/fileManager001.txt']
}

config = ps.ensemble.file_manager_product(managers_to_run)
pprint(config)
print(config['file_manager'])
sim = ps.Ensemble(config['summa_exe'],run_config, config['file_manager'],num_workers=8)
print(sim)
sim.run('local')
(sim.summary())

{'001': {'forcing_file_list': '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/forcing/forcing_files_001.txt'}, '004': {'forcing_file_list': '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/forcing/forcing_files_004.txt'}, '003': {'forcing_file_list': '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/forcing/forcing_files_003.txt'}, '002': {'forcing_file_list': '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/forcing/forcing_files_002.txt'}, '005': {'forcing_file_list': '/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/forcing/forcing_files_005.txt'}}
/Users/dcasson/Data/summa_snakemake/bow_above_banff/summa/settings/fileManager.txt


{'Success': [], 'Error': ['001', '004', '003', '002', '005'], 'Other': []}

In [ ]:
import pysumma as ps
import subprocess

cmd = """
ml load gcc/14.2.0
ml load openblas/0.3.17
ml load netcdf-fortran/4.5.3
"""
subprocess.run(cmd, shell=True, executable="/bin/bash")

config = {
    'summa_exe': '/home/x-dcasson/GitRepos/summa/bin/summa_sundials.exe',
    'file_manager': '/anvil/projects/x-ees240082/users/dcasson/gpep/tuolumne/summa/settings/.pysumma/003/fileManager.txt',
    'run_suffix': '006',
    'gru': 25
}
sim = ps.Simulation(config['summa_exe'], config['file_manager'])
sim.run(run_suffix=config['run_suffix'], startGRU=int(config['gru']), countGRU=1, write_config=False)
print(sim.status)


Inactive Modules:
  1) openmpi/4.0.6

Due to MODULEPATH changes, the following have been reloaded:
  1) numactl/2.0.14     2) zlib/1.2.11

The following have been reloaded with a version change:
  1) gcc/11.2.0 => gcc/14.2.0           3) mpc/1.1.0 => mpc/1.3.1-7jkixwl
  2) gmp/6.2.1 => gmp/6.3.0-kn4kkkg     4) mpfr/4.0.2 => mpfr/4.2.1-xvyu75c



//home/x-dcasson/GitRepos/summa/bin/summa_sundials.exe: error while loading shared libraries: libnetcdf.so.18: cannot open shared object file: No such file or directory

